# Part 1: Exploring the dataset

You will now explore the dataset a little bit. You will first get a feel of what the data looks like. You will print some of the data and learn how to tokenize the sentences in the data to individual words. For the English language, tokenization appears to be a trivial task, however, there are languages such as Japanese, which are not as consistently delimited as English.

In [1]:
import pandas as pd
en_text = pd.read_csv('dataset/vocab_en.txt', header=None, delimiter='\n')
# en_text.head()
fr_text = pd.read_csv('dataset/vocab_fr.txt', header=None, delimiter='\n')
# fr_text.head()
en_text = en_text.iloc[:,0].values.tolist()
fr_text = fr_text.iloc[:,0].values.tolist()


In [2]:
# Iterate through the first 5 English and French sentences in the dataset
for en_sent, fr_sent in zip(en_text[:5], fr_text[:5]):  
  print("English: ", en_sent)
  print("\tFrench: ", fr_sent)

# Get the first sentence of the English dataset
first_sent = en_text[0]
print("First sentence: ", first_sent)
# Tokenize the first sentence
first_words = first_sent.split(" ")
# Print the tokenized words
print("\tWords: ", first_words)

English:  new jersey is sometimes quiet during autumn , and it is snowy in april .
	French:  new jersey est parfois calme pendant l' automne , et il est neigeux en avril .
English:  the united states is usually chilly during july , and it is usually freezing in november .
	French:  les états-unis est généralement froid en juillet , et il gèle habituellement en novembre .
English:  california is usually quiet during march , and it is usually hot in june .
	French:  california est généralement calme en mars , et il est généralement chaud en juin .
English:  the united states is sometimes mild during june , and it is cold in september .
	French:  les états-unis est parfois légère en juin , et il fait froid en septembre .
English:  your least liked fruit is the grape , but my least liked is the apple .
	French:  votre moins aimé fruit est le raisin , mais mon moins aimé est la pomme .
First sentence:  new jersey is sometimes quiet during autumn , and it is snowy in april .
	Words:  ['new',

# Part 2: Exploring the dataset

Now you will explore some attributes of the dataset. Specifically, you will determine the average length (i.e. number of words) of all sentences and the size of the vocabulary for the English dataset.

For this exercise, the English dataset en_text containing a list of English sentences has been provided. In this exercise you will be using a Python list-related function called <list>.extend() which is a different variant of the function <list>.append(). Let's understand the difference through an example. Say a=[1,2,3] and b=[4,5]. a.append(b) would result in a list [1,2,3,[4,5]] where a.extend(b) would result in [1,2,3,4,5]

In [3]:
import numpy as np
# Compute length of sentences
sent_lengths = [len(en_sent.split(" ")) for en_sent in en_text]
# Compute the mean of sentences lengths
mean_length = np.mean(sent_lengths)
print('(English) Mean sentence length: ', mean_length)

all_words = []
for sent in en_text:
  # Populate all_words with all the words in sentences
  all_words.extend(sent.split(" "))
# Compute the length of the set containing all_words
vocab_size = len(set(all_words))
print("(English) Vocabulary size: ", vocab_size)

(English) Mean sentence length:  13.225678224285508
(English) Vocabulary size:  228


# Defining the encoder

Here you'll be taking your first step towards creating a machine translation model: implementing the encoder. The encoder that you will implement is a very simple model compared to the complex models that are used in real-world applications such as the Google machine translation service. But don't worry, though the model is simple, the concepts are the same as of those complex models. Here we will use the prefix en (e.g. en_gru) to indicate anything encoder related and de to indicate decoder related things (e.g. de_gru).

You will see that we are choosing en_vocab to be smaller (150) than the actual value (228) that we found. Making the vocabulary smaller reduces the memory footprint of the model. Reducing the vocabulary slightly is fine as we are removing the rarest words when we are doing so. For machine translation tasks, rare words usually have less value than common words.

In [4]:
import tensorflow.keras as keras

en_len = 15
en_vocab = 150
hsize = 48

# Define an input layer
en_inputs = keras.layers.Input(shape=(en_len,en_vocab))
# Define a GRU layer which returns the state
en_gru = keras.layers.GRU(hsize, return_state=True)
# Get the output and state from the GRU
output, state = en_gru(en_inputs)
# Define and print the model summary
encoder = keras.models.Model(inputs=en_inputs, outputs=state)
print(encoder.summary())

Model: "functional_1"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_1 (InputLayer)         [(None, 15, 150)]         0         
_________________________________________________________________
gru (GRU)                    [(None, 48), (None, 48)]  28800     
Total params: 28,800
Trainable params: 28,800
Non-trainable params: 0
_________________________________________________________________
None


# Understanding the RepeatVector layer

You will now explore how the RepeatVector layer works. The RepeatVector layer adds an extra dimension to your dataset. For example if you have an input of shape (batch size, input size) and you want to feed that to a GRU layer, you can use a RepeatVector layer to convert the input to a tensor with shape (batch size, sequence length, input size).

In this exercise, you will define a model that repeats a given input a fixed number of times. You will then feed a numpy array to the model and investigate how the model changes the output.

In [5]:
from tensorflow.keras.layers import Input, RepeatVector
from tensorflow.keras.models import Model
import numpy as np

inp = Input(shape=(2,))
# Define a RepeatVector that repeats the input 6 times
rep = RepeatVector(6)(inp)
# Define a model
model = Model(inputs=inp, outputs=rep)
# Define input x
x = np.array([[0,1], [2,3]])
# Get model prediction y
y = model.predict(x)
print('x.shape = ',x.shape,'\ny.shape = ',y.shape)

x.shape =  (2, 2) 
y.shape =  (2, 6, 2)


# The shape of a RepeatVector layer output

Consider the following usage of the RepeatVector layer
```
inp = Input(shape=(3,))
rep = RepeatVector(10)(inp)
model = Model(inputs=inp, outputs=rep)
```
If you pass x of shape 8x3 to the model.predict() function, what is the shape of the output?

In [6]:
inp = Input(shape=(3,))
rep = RepeatVector(10)(inp)
temp_model = Model(inputs=inp, outputs=rep)
x_test = np.random.rand(8, 3)
y_test = temp_model.predict(x_test)
y_test.shape

(8, 10, 3)

# Defining the decoder

In this exercise, you will implement the decoder and define an end-to-end model going from encoder inputs to the decoder GRU outputs. The decoder uses the same model as the encoder. However there are differences in the inputs and states fed to the decoder, compared to the encoder. For example, the decoder consumes the context vector produced by the encoder as inputs as well as the initial state to the decoder. Remember that we will use the prefix en (e.g. en_gru) to indicate anything encoder related and de to indicate decoder related things (e.g. de_gru).

To implement the decoder you will use RepeatVector and GRU layers.

For this exercise you have been provided with the encoder model and the various layers of the encoder that you have already implemented. For example, the encoder inputs are provided as en_inputs and the context vector as en_state. Also note that the GRU and Model objects have been already imported.

In [7]:
en_state = state

In [8]:

import keras
from tensorflow.keras.layers import RepeatVector

hsize = 48
fr_len = 20
# Define a RepeatVector layer
de_inputs = RepeatVector(fr_len)(en_state)
# Define a GRU model that returns all outputs
decoder_gru = keras.layers.GRU(hsize, return_sequences=True)
# Get the outputs of the decoder
gru_outputs = decoder_gru(de_inputs , initial_state=en_state)
# Define a model with the correct inputs and outputs
enc_dec = Model(inputs=en_inputs, outputs=gru_outputs)
enc_dec.summary()

Model: "functional_7"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 15, 150)]    0                                            
__________________________________________________________________________________________________
gru (GRU)                       [(None, 48), (None,  28800       input_1[0][0]                    
__________________________________________________________________________________________________
repeat_vector_2 (RepeatVector)  (None, 20, 48)       0           gru[0][1]                        
__________________________________________________________________________________________________
gru_1 (GRU)                     (None, 20, 48)       14112       repeat_vector_2[0][0]            
                                                                 gru[0][1]             

# Part 1: Enter to win amazing prizes

In this exercise, you will learn about the Dense layer. Why not do that with a fun exercise? Imagine there's a game show where prizes are determined by a neural network. The contestant enters

- the number of siblings,
- the number of coffees had today and
- if they like tomatoes or not,
- and the model predicts what the contestant will win.

To implement this, you will be using Keras. You will need to create a model with an input layer which accepts three features (the number of siblings as an integer, the number of coffees as an integer and if they like tomatoes or not as a 0 or 1). Then the input goes through a Dense layer which outputs 3 probabilities (i.e. probabilities of winning a car, a gift voucher or nothing).

In [9]:
from keras.initializers import RandomNormal
from keras.layers import Dense
init = RandomNormal()
print(init)


In [10]:
# Define an input layer with batch size 3 and input size 3
inp = Input(shape=(3,3))
# Get the output of the 3 node Dense layer
pred = Dense(3, activation='softmax', kernel_initializer=init, bias_initializer=init)(inp)
model = Model(inputs=inp, outputs=pred)

names = ["Mark", "John", "Kelly"]
prizes = ["Gift voucher", "Car", "Nothing"]
x = np.array([[5, 0, 1], [0, 3, 1], [2, 2, 1]])
# Compute the model prediction for x
y = model.predict(x)
# Get the most probable class for each sample
classes = np.argmax(y, axis=1)
print("\n".join(["{} has probabilities {} and wins {}".format(n,p,prizes[c]) \
                 for n,p,c in zip(names, y, classes)]))

Mark has probabilities [0.38767925 0.33051768 0.2818031 ] and wins Gift voucher
John has probabilities [0.3182557  0.28111154 0.4006328 ] and wins Nothing
Kelly has probabilities [0.3459771  0.29959404 0.35442886] and wins Nothing


# Part 2: Let's play a few more games

Great work on the last project. This time, you will need to simulate multiple game shows hosted over several days. This means that your data will have a time dimension to it. More specifically, your data will have the shape (number of contestants, game shows, inputs size).

You will need to extend your model to incorporate this new feature. For this you will be using a TimeDistributed layer to allow the Dense layer to accept contestants from multiple game shows.

You have been provided with the weight initializer init, the prizes list from the previous exercise, a time-series input x and names which contains the names of the contestants. x is a (3,2,3) numpy array where names is a (2,3) Python list. In other words, you have 2 game shows (i.e. sequence length), each with 3 contestants (batch size) where each contestant has 3 attributes (input size).

In [11]:
# Print names and x
print('names=\n',names, '\nx=\n',x, '\nx.shape=', x.shape)

names=
 ['Mark', 'John', 'Kelly'] 
x=
 [[5 0 1]
 [0 3 1]
 [2 2 1]] 
x.shape= (3, 3)


In [12]:
print(*zip(names, y, classes))
print(*zip(names, y[:,1], classes))
x = np.array([[[5, 0, 1],
        [1, 1, 0]],

       [[0, 3, 1],
        [0, 4, 0]],

       [[2, 2, 1],
        [6, 0, 1]]])
x.shape

('Mark', array([0.38767925, 0.33051768, 0.2818031 ], dtype=float32), 0) ('John', array([0.3182557 , 0.28111154, 0.4006328 ], dtype=float32), 2) ('Kelly', array([0.3459771 , 0.29959404, 0.35442886], dtype=float32), 2)
('Mark', 0.33051768, 0) ('John', 0.28111154, 2) ('Kelly', 0.29959404, 2)


(3, 2, 3)

In [13]:
from keras.layers import TimeDistributed

inp = Input(shape=(2, 3))
# Create the TimeDistributed layer (the output of the Dense layer)
dense_time = TimeDistributed(Dense(3, activation='softmax', kernel_initializer=init, bias_initializer=init))
pred = dense_time(inp)
model = Model(inputs=inp, outputs=pred)

y = model.predict(x)
# Get the most probable class for each sample
classes = np.argmax(y, axis=-1)
for t in range(2):
  # Get the t-th time-dimension slice of y and classes
  for n, p, c in zip(names[t], y[:, t, :], classes[:, t]):
  	print("Game {}: {} has probs {} and wins {}\n".format(t+1,n,p,prizes[c]))

Game 1: M has probs [0.4200121  0.23044024 0.3495477 ] and wins Gift voucher

Game 1: a has probs [0.32310838 0.28717503 0.38971657] and wins Nothing

Game 1: r has probs [0.35872093 0.2627058  0.3785733 ] and wins Nothing

Game 2: J has probs [0.35450113 0.27948588 0.36601296] and wins Nothing

Game 2: o has probs [0.3134227  0.2699953  0.41658202] and wins Nothing

Game 2: h has probs [0.433811   0.21569622 0.35049278] and wins Gift voucher



# Part 1: Defining the full model

Here you will be implementing the last few layers of the encoder-decoder model. You will be using Dense and TimeDistributed layers to get the final predictions (i.e. predicted French word probabilities) of the encoder-decoder model.

You are provided with the encoder and decoder (without the top-part) you implemented so far. The decoder GRU layer's output de_out is provided. We use the prefix en (e.g. en_gru) to indicate anything encoder related and de to indicate decoder related things (e.g. de_gru).

In [14]:
fr_vocab = 250

from keras.layers import GRU, Input
from keras.models import Model


# Create input layer
# input_layer = Input(shape=(25, 100))
en_inputs = Input(shape=(25, 100))


# Create GRU layer
gru_layer = GRU(10, return_sequences=True, name='gru_1')
de_out = gru_layer(en_inputs)


In [15]:
# Import Dense and TimeDistributed layers
from tensorflow.keras.layers import Dense, TimeDistributed
# Define a softmax dense layer that has fr_vocab outputs
de_dense = Dense(fr_vocab, activation='softmax')
# Wrap the dense layer in a TimeDistributed layer
de_dense_time = TimeDistributed(de_dense)
# Get the final prediction of the model
de_pred = de_dense_time(de_out)
print("Prediction shape: ", de_pred.shape)

Prediction shape:  (None, 25, 250)


# Part 2: Defining the full model

Did you know that it took around 6 days and 96 GPUs to train a variant of the Google Neural Machine Translator just on the English to French translation task.

In this exercise you will define a similar but much simpler encoder-decoder based neural machine translator model. Specifically, you will use the previously defined inputs and outputs and define a Keras Model object and compile the model with a given loss function and an optimizer.

Here you are provided with en_inputs (encoder input layer), en_out and en_state (encoder GRU output), de_out (decoder GRU output) and de_pred (decoder prediction) that you previously defined.

In [16]:
from tensorflow.keras.models import Model
# Define a model with encoder input and decoder output
nmt = Model(inputs=en_inputs, outputs=de_pred)

# Compile the model with an optimizer and a loss
nmt.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['acc'])

# View the summary of the model 
nmt.summary()

Model: "functional_13"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_6 (InputLayer)         [(None, 25, 100)]         0         
_________________________________________________________________
gru_1 (GRU)                  (None, 25, 10)            3360      
_________________________________________________________________
time_distributed_1 (TimeDist (None, 25, 250)           2750      
Total params: 6,110
Trainable params: 6,110
Non-trainable params: 0
_________________________________________________________________
